In [12]:
!pip install -q transformers torch scikit-learn pandas numpy accelerate
print("All installed!")

All installed!


In [13]:
import torch
import pandas as pd
import numpy as np
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.nn import CrossEntropyLoss
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("All imports done! ✅")

Device: cuda
GPU: Tesla T4
All imports done! ✅


In [3]:
from google.colab import files
print("Upload train_df.csv, val_df.csv and label_mapping.json")
uploaded = files.upload()
print("Uploaded! ✅")

Upload train_df.csv, val_df.csv and label_mapping.json


Saving label_mapping.json to label_mapping (1).json
Saving train_df.csv to train_df (1).csv
Saving val_df.csv to val_df (1).csv
Uploaded! ✅


In [4]:
train_df = pd.read_csv('train_df.csv')
val_df = pd.read_csv('val_df.csv')

with open('label_mapping.json', 'r') as f:
    mapping = json.load(f)

label2id = mapping['label2id']
id2label = {int(k): v for k, v in mapping['id2label'].items()}

print("Train size:", len(train_df))
print("Val size:", len(val_df))
print("Labels:", label2id)
print("Data loaded! ✅")

Train size: 25011
Val size: 6253
Labels: {'FACTUAL': 0, 'UNCERTAIN': 1, 'HALLUCINATION': 2}
Data loaded! ✅


In [5]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

def tokenize_data(df, max_length=256):
    return tokenizer(
        df['premise'].astype(str).tolist(),
        df['hypothesis'].astype(str).tolist(),
        truncation=True,
        max_length=max_length,
        padding='max_length',
        return_tensors='pt'
    )

print("Tokenizing training set...")
train_encodings = tokenize_data(train_df)
print("Tokenizing validation set...")
val_encodings = tokenize_data(val_df)
print("Tokenization done! ✅")

Tokenizing training set...
Tokenizing validation set...
Tokenization done! ✅


In [6]:
class HallucinationDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = HallucinationDataset(train_encodings, train_df['label_id'].tolist())
val_dataset = HallucinationDataset(val_encodings, val_df['label_id'].tolist())

print("Train dataset:", len(train_dataset))
print("Val dataset:", len(val_dataset))
print("Datasets ready! ✅")

Train dataset: 25011
Val dataset: 6253
Datasets ready! ✅


In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
    torch_dtype=torch.float32
)

# Force all parameters to float32
model = model.float()
model = model.to(device)

# Verify dtype
sample_param = next(model.parameters())
print(f"Model dtype: {sample_param.dtype}")

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded! ✅")
print(f"Total parameters: {total_params:,}")
print(f"Model on: {device}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.den

Model dtype: torch.float32
Model loaded! ✅
Total parameters: 184,424,451
Model on: cuda


In [8]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    f1_macro = f1_score(labels, predictions, average='macro')
    return {
        'accuracy': round(accuracy, 4),
        'f1_weighted': round(f1, 4),
        'f1_macro': round(f1_macro, 4)
    }
print("Metrics function ready! ✅")

Metrics function ready! ✅


In [9]:
training_args = TrainingArguments(
    output_dir='./deberta_hallucination',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    warmup_steps=200,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    greater_is_better=True,
    fp16=False,
    bf16=False,
    report_to="none"
)
print("Training arguments set! ✅")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Batch size: {training_args.per_device_train_batch_size}")

Training arguments set! ✅
Learning rate: 2e-05
Batch size: 8


In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Verify loss before training
print("Checking loss on sample batch...")
sample_batch = next(iter(torch.utils.data.DataLoader(train_dataset, batch_size=4)))
sample_batch = {k: v.to(device) for k, v in sample_batch.items()}
labels = sample_batch.pop('labels')
with torch.no_grad():
    outputs = model(**sample_batch)
    logits = outputs.logits
    loss = torch.nn.CrossEntropyLoss()(logits.float(), labels)
    print(f"Sample loss: {loss.item():.4f}")
    print(f"Logits dtype: {logits.dtype}")

if loss.item() > 0.5:
    print("\n✅ Loss looks good — starting training!")
    trainer.train()
    print("\nTraining complete! ✅")
else:
    print("\n❌ Loss still wrong — do not train!")

Checking loss on sample batch...
Sample loss: 1.0086
Logits dtype: torch.float32

✅ Loss looks good — starting training!


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted,F1 Macro
1,0.375199,0.453333,0.863300,0.862500,0.856600
2,0.337251,0.486620,0.866300,0.866500,0.860700
3,0.216118,0.582258,0.870500,0.869000,0.862700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete! ✅


In [11]:
model.save_pretrained('./hallucination_model')
tokenizer.save_pretrained('./hallucination_model')

print("Model saved! ✅")

import shutil
shutil.make_archive('hallucination_model', 'zip', './hallucination_model')
print("Model zipped! ✅")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved! ✅
Model zipped! ✅


In [14]:
import os

files = os.listdir('./hallucination_model') if os.path.exists('./hallucination_model') else []
print("Model files found:", files)

Model files found: ['config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import login
from transformers import AutoModelForSequenceClassification, AutoTokenizer

login(token="YOUR_HF_TOKEN_HERE")

# Load saved model
model = AutoModelForSequenceClassification.from_pretrained('./hallucination_model')
tokenizer = AutoTokenizer.from_pretrained('./hallucination_model')

# Push using correct username
model.push_to_hub("NiviG/hallucination-detector")
tokenizer.push_to_hub("NiviG/hallucination-detector")

print("✅ Model uploaded!")
print("URL: https://huggingface.co/NiviG/hallucination-detector")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...2dlm5rp/model.safetensors:   0%|          |  553kB /  738MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

✅ Model uploaded!
URL: https://huggingface.co/NiviG/hallucination-detector
